**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to FPGA

> ⚠️ **Draft — code not machine-verified.** Verilog below is shown in fenced blocks and has not been simulated by an automated check. Before teaching, an instructor should run each module + testbench once in a simulator (Icarus Verilog is free: `iverilog -o sim tb.v mod.v && vvp sim`, or use [EDA Playground](https://edaplayground.com/) in a browser). Remove this banner after that pass.

Where [GPU workshops](../Intro_GPU/README.md) parallelize *software* across fixed hardware, an FPGA lets you **build the datapath itself**: your filter becomes wires, registers, and multipliers with sample-rate throughput and microsecond-class latency. This is how radar front-ends, SDRs, and instrument DSP actually ship.

## 0. Introduction

An FPGA (*Field-Programmable Gate Array*) is a sea of small configurable pieces:

- **LUTs** (look-up tables) — implement any small boolean function;
- **Flip-flops** — 1-bit registers; state lives here between clock edges;
- **DSP slices** — hard multiply-accumulate blocks (your FIR taps);
- **Block RAM** — on-chip memory; **routing fabric** — programmable wiring connecting it all.

"Programming" an FPGA = describing hardware in an HDL, then letting tools map it onto these resources.

## 1. Pre-requisites

- [Intro to C](../Intro_Programming/Intro_C.ipynb) — bits, twos-complement, thinking in memory.
- [Filter Design](../Intro_DSP/Filter_Design.ipynb) — Session 3 implements that FIR here.
- Tools: any simulator (Icarus/EDA Playground) for S1–S3; a dev board (e.g. Basys 3, iCEBreaker) + vendor toolchain only for S4.

---
### 🕐 Session 1 of 4 — *What Is an FPGA?* (~35 min)
**Goal:** LUTs, flip-flops, DSP slices; the FPGA vs CPU vs GPU trade-space.
**Feeds into:** Session 2 (HDL basics).

---

## 2. The Trade-Space

💡 **Intuition.** A CPU executes your algorithm *over time* — one flexible pipeline reused every instruction. An FPGA lays your algorithm out *in space* — every operation gets its own silicon, all active every clock cycle. That's why a modest 200 MHz FPGA can out-throughput a 5 GHz CPU on streaming DSP: a 64-tap filter does 64 multiplies *simultaneously*, forever, with no instruction fetch at all.

| | CPU | GPU | FPGA |
|---|---|---|---|
| Flexibility | highest | high | rebuild to change |
| Latency | µs–ms, jittery ([OS scheduling!](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb)) | high (batching) | **deterministic, cycles** |
| Throughput/W on streaming DSP | low | mid | **highest** |
| Dev effort | low | mid | high |

---
### 🕐 Session 2 of 4 — *HDL Basics* (~40 min)
**Goal:** modules, combinational vs sequential logic; simulate a counter with a testbench.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (a hardware FIR).

---

## 3. Verilog: Describing, Not Instructing

💡 **Intuition.** HDL code is not a program — it's a **circuit diagram in text**. Every `assign` is a wire that exists *always*; every clocked `always` block is a row of flip-flops. Nothing "runs top to bottom"; everything happens at once. Unlearning sequential-execution instinct is the entire difficulty of week one.

**Combinational** logic (no memory — outputs follow inputs like wiring):

```verilog
module mux2 (input  wire a, b, sel,
             output wire y);
  assign y = sel ? b : a;      // a physical multiplexer, not an "if that runs"
endmodule
```

**Sequential** logic (state changes only on the clock edge):

```verilog
module counter #(parameter W = 8) (
    input  wire         clk, rst,
    output reg  [W-1:0] count
);
  always @(posedge clk) begin
    if (rst) count <= 0;
    else     count <= count + 1;   // <= is the *non-blocking* clocked assignment
  end
endmodule
```

**Testbench** — the simulation driver (never synthesized):

```verilog
`timescale 1ns/1ps
module tb_counter;
  reg clk = 0, rst = 1;
  wire [7:0] count;
  counter dut (.clk(clk), .rst(rst), .count(count));

  always #5 clk = ~clk;                 // 100 MHz clock
  initial begin
    $dumpfile("counter.vcd"); $dumpvars(0, tb_counter);
    #12 rst = 0;                        // release reset off the edge
    #200 $display("count = %d (expect ~20)", count);
    $finish;
  end
endmodule
```

Run: `iverilog -o sim tb_counter.v counter.v && vvp sim` — then open `counter.vcd` in GTKWave and
*look at the waveform*. Hardware debugging is waveform reading.

---
### 🕐 Session 3 of 4 — *A Hardware FIR Filter* (~40 min)
**Goal:** implement the [Filter Design](../Intro_DSP/Filter_Design.ipynb) FIR as fixed-point hardware; understand pipelining.
**Builds on:** Session 2.

---

## 4. The FIR, in Silicon

💡 **Intuition.** An FIR filter is *born* hardware-shaped: the tapped delay line is a shift register (flip-flops), each tap a DSP-slice multiply, the sum an adder tree. And since there's no float unit in the fabric, taps become **fixed-point** integers: scale by $2^{15}$, multiply, shift back — Q1.15 arithmetic. Quantizing taps moves the stopband floor; check the quantized response in Python *before* burning it into silicon.

```verilog
// 4-tap FIR, Q1.15 coefficients, one output per clock (transposed form)
module fir4 (
    input  wire               clk, rst,
    input  wire signed [15:0] x_in,     // Q1.15 sample
    output reg  signed [15:0] y_out
);
  // taps from your Python design, scaled: round(h * 2^15)
  localparam signed [15:0] H0 = 16'sd3277,  H1 = 16'sd13107,
                           H2 = 16'sd13107, H3 = 16'sd3277;   // ≈ [0.1 0.4 0.4 0.1]

  reg signed [33:0] acc [0:3];          // 16x16 products need 32 bits + growth
  integer i;
  always @(posedge clk) begin
    if (rst) begin
      for (i = 0; i < 4; i = i + 1) acc[i] <= 0;
      y_out <= 0;
    end else begin
      acc[3] <= x_in * H3;                       // transposed FIR: partial sums
      acc[2] <= x_in * H2 + acc[3];              //   flow through registers —
      acc[1] <= x_in * H1 + acc[2];              //   pipelining is built in
      acc[0] <= x_in * H0 + acc[1];
      y_out  <= acc[0] >>> 15;                   // Q2.30 → Q1.15 (arithmetic shift)
    end
  end
endmodule
```

Verification strategy (the professional habit): generate a test input **and golden output in
Python** with the *same* quantized taps, feed the input to the testbench, and assert the hardware
matches sample-for-sample. Your notebook is the reference model; the waveform is the proof.

---
### 🕐 Session 4 of 4 — *Toolchain & Deployment* (~35 min)
**Goal:** synthesis → place & route → timing closure; blink a real board.
**Builds on:** Sessions 2–3.

---

## 5. From Text to Silicon

The flow every vendor tool implements:

1. **Synthesis** — Verilog → netlist of LUTs/FFs/DSPs.
2. **Place & route** — assign each element a physical site; find wiring.
3. **Timing analysis** — is every register-to-register path faster than the clock period? If not (*negative slack*), pipeline more or slow the clock. This is where the transposed FIR's built-in registers pay off.
4. **Bitstream** — the configuration file loaded onto the chip.

💡 **Intuition.** *Timing closure* is the hardware version of the [uncertainty principle trade-offs](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) you've seen all curriculum: combinational depth (work per cycle) trades against clock frequency (cycles per second). Pipelining buys frequency with latency.

First-board checklist: vendor tool (Vivado for Basys 3; open-source yosys+nextpnr for iCE40), the board's constraint file mapping pins, and the traditional first design — the counter from Session 2 driving LEDs.

## 6. Conclusion

FPGAs lay algorithms out in space: LUTs and flip-flops describe logic, HDL describes circuits (not steps), fixed-point FIRs map perfectly onto DSP slices, and timing closure is the price of speed. You now know the full path from `scipy.signal.firwin` to a bitstream.

---
## Where next

- [Filter Design](../Intro_DSP/Filter_Design.ipynb) — design and quantize the taps you'll synthesize.
- [Intro to GPU Systems](../Intro_GPU/README.md) — the other acceleration path, compared honestly in Session 1.
- [Intro to C](../Intro_Programming/Intro_C.ipynb) — the software still driving the FPGA from the host side.